# Pollutants Data Pre-Processing

This notebook performs comprehensive pre-processing of global air pollutant datasets, including:

1. Loading multiple pollutant datasets (HAP, NO2, OZONE, PM25)
2. Cleaning data formatting artifacts
3. Analyzing dataset structure and content
4. Removing unnecessary columns
5. Standardizing column names with pollutant-specific suffixes
6. Merging datasets into a consolidated format for analysis
7. Detecting and handling missing values
8. Converting and validating appropriate data types
9. Implementing time-aware outlier detection methods
10. Identifying and resolving duplicate records
11. Exporting the processed dataset for further analysis

## 1. Import Necessary Libraries

For this analysis, we'll use pandas for data manipulation, cleaning, and analysis.

In [1]:
!pip install pandas numpy scipy matplotlib

zsh:1: command not found: pip


In [2]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

## 2. Load Pollutant Datasets

We'll load four distinct pollutant datasets:
1. **HAP** (Household Air Pollution) 
2. **NO2** (Nitrogen Dioxide)
3. **OZONE**
4. **PM2.5** (Particulate Matter 2.5 micrometers or smaller)

In [3]:
# Load HAP (Household Air Pollution) dataset
file_path = '../Datasets/GLOBAL_POLLUTANT_DATA_HAP.csv'
pollutant_data_hap = pd.read_csv(file_path)
print(f"Loaded HAP dataset with shape: {pollutant_data_hap.shape}")

# Load NO2 (Nitrogen Dioxide) dataset
file_path = '../Datasets/GLOBAL_POLLUTANT_DATA_NO2.csv'
pollutant_data_no2 = pd.read_csv(file_path)
print(f"Loaded NO2 dataset with shape: {pollutant_data_no2.shape}")

# Load OZONE dataset
file_path = '../Datasets/GLOBAL_POLLUTANT_DATA_OZONE.csv'
pollutant_data_ozone = pd.read_csv(file_path)
print(f"Loaded OZONE dataset with shape: {pollutant_data_ozone.shape}")

# Load PM25 (Particulate Matter 2.5) dataset
file_path = '../Datasets/GLOBAL_POLLUTANT_DATA_PM25.csv'
pollutant_data_pm25 = pd.read_csv(file_path)
print(f"Loaded PM25 dataset with shape: {pollutant_data_pm25.shape}")

Loaded HAP dataset with shape: (6357, 14)
Loaded NO2 dataset with shape: (6357, 14)
Loaded OZONE dataset with shape: (6357, 14)
Loaded PM25 dataset with shape: (6357, 14)


In [4]:
# Display the head of all 4 datasets side by side
pd.concat(
    [pollutant_data_hap.head(), pollutant_data_no2.head(), pollutant_data_ozone.head(), pollutant_data_pm25.head()],
    keys=['HAP', 'NO2', 'OZONE', 'PM25'],
    names=['Dataset']
)

="Exposure Id"     ="Type"      ="Country" ="ISO3"  ="Region"  \
Dataset                                                                    
HAP     0        175151.0  ="country"  ="Afghanistan"  ="AFG"  ="region"   
        1        175152.0  ="country"  ="Afghanistan"  ="AFG"  ="region"   
        2        175153.0  ="country"  ="Afghanistan"  ="AFG"  ="region"   
        3        175154.0  ="country"  ="Afghanistan"  ="AFG"  ="region"   
        4        175155.0  ="country"  ="Afghanistan"  ="AFG"  ="region"   
NO2     0        175182.0  ="country"  ="Afghanistan"  ="AFG"  ="region"   
        1        175190.0  ="country"  ="Afghanistan"  ="AFG"  ="region"   
        2        175191.0  ="country"  ="Afghanistan"  ="AFG"  ="region"   
        3        175192.0  ="country"  ="Afghanistan"  ="AFG"  ="region"   
        4        175193.0  ="country"  ="Afghanistan"  ="AFG"  ="region"   
OZONE   0        175120.0  ="country"  ="Afghanistan"  ="AFG"  ="region"   
        1        175121.0  ="country"  ="Afghanistan"  ="AFG"  ="region"   
        2        175122.0  ="country"  ="Afghanistan"  ="AFG"  ="region"   
        3        175123.0  ="country"  ="Afghanistan"  ="AFG"  ="region"   
        4        175124.0  ="country"  ="Afghanistan"  ="AFG"  ="region"   
PM25    0        175089.0  ="country"  ="Afghanistan"  ="AFG"  ="region"   
        1        175104.0  ="country"  ="Afghanistan"  ="AFG"  ="region"   
        2        175105.0  ="country"  ="Afghanistan"  ="AFG"  ="region"   
        3        175106.0  ="country"  ="Afghanistan"  ="AFG"  ="region"   
        4        175107.0  ="country"  ="Afghanistan"  ="AFG"  ="region"   

                                   ="Name" ="Exposure Lower" ="Exposure Mean"  \
Dataset                                                                         
HAP     0  ="North Africa and Middle East"           ="0.80"          ="0.86"   
        1  ="North Africa and Middle East"           ="0.80"          ="0.86"   
        2  ="North Africa and Middle East"           ="0.80"          ="0.86"   
        3  ="North Africa and Middle East"           ="0.80"          ="0.86"   
        4  ="North Africa and Middle East"           ="0.80"          ="0.86"   
NO2     0  ="North Africa and Middle East"           ="0.95"          ="12.3"   
        1  ="North Africa and Middle East"          ="0.806"          ="12.1"   
        2  ="North Africa and Middle East"          ="0.636"          ="11.7"   
        3  ="North Africa and Middle East"          ="0.443"          ="11.4"   
        4  ="North Africa and Middle East"           ="0.23"            ="11"   
OZONE   0  ="North Africa and Middle East"           ="55.8"          ="56.6"   
        1  ="North Africa and Middle East"           ="54.6"          ="55.4"   
        2  ="North Africa and Middle East"           ="53.2"          ="54.1"   
        3  ="North Africa and Middle East"           ="52.7"          ="53.6"   
        4  ="North Africa and Middle East"           ="53.1"            ="54"   
PM25    0  ="North Africa and Middle East"           ="32.1"          ="64.2"   
        1  ="North Africa and Middle East"           ="36.9"          ="64.2"   
        2  ="North Africa and Middle East"           ="39.5"          ="64.2"   
        3  ="North Africa and Middle East"           ="40.5"          ="64.2"   
        4  ="North Africa and Middle East"           ="39.8"          ="64.3"   

          ="Exposure Upper"  ="Year" ="Pollutant"    ="Pollutant Name"  \
Dataset                                                                  
HAP     0           ="0.90"  ="1990"       ="hap"               ="HAP"   
        1           ="0.90"  ="1991"       ="hap"               ="HAP"   
        2           ="0.90"  ="1992"       ="hap"               ="HAP"   
        3           ="0.90"  ="1993"       ="hap"               ="HAP"   
        4           ="0.90"  ="1994"       ="hap"               ="HAP"   
NO2     0           ="12.3"  ="1990" 

## 3. Data Cleaning Process

### 3.1 Addressing Excel-style Formatting Artifacts

In all of the pollutant data files, we observe unnecessary Excel-style formatting artifacts, particularly the `="..."` pattern in both column headers and cell values. This formatting needs to be removed for proper data analysis.

In [5]:
# Define function to clean dataset format issues
def clean_excel_artifacts(df):
    """
    Clean column headers and cell values by removing Excel-style formatting artifacts (="...")
    """
    # Clean column headers
    df.columns = [col.strip().strip('="') for col in df.columns]
    
    # Clean each column using .map() if it's of string/object dtype
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].map(lambda x: x.strip('="') if isinstance(x, str) else x)
    
    return df

# Clean each dataset individually
pollutant_data_hap = clean_excel_artifacts(pollutant_data_hap)
pollutant_data_no2 = clean_excel_artifacts(pollutant_data_no2)
pollutant_data_ozone = clean_excel_artifacts(pollutant_data_ozone)
pollutant_data_pm25 = clean_excel_artifacts(pollutant_data_pm25)

# Display the head of all 4 datasets side by side
pd.concat(
    [pollutant_data_hap.head(), pollutant_data_no2.head(), pollutant_data_ozone.head(), pollutant_data_pm25.head()],
    keys=['HAP', 'NO2', 'OZONE', 'PM25'],
    names=['Dataset']
)

Exposure Id     Type      Country ISO3  Region  \
Dataset                                                     
HAP     0     175151.0  country  Afghanistan  AFG  region   
        1     175152.0  country  Afghanistan  AFG  region   
        2     175153.0  country  Afghanistan  AFG  region   
        3     175154.0  country  Afghanistan  AFG  region   
        4     175155.0  country  Afghanistan  AFG  region   
NO2     0     175182.0  country  Afghanistan  AFG  region   
        1     175190.0  country  Afghanistan  AFG  region   
        2     175191.0  country  Afghanistan  AFG  region   
        3     175192.0  country  Afghanistan  AFG  region   
        4     175193.0  country  Afghanistan  AFG  region   
OZONE   0     175120.0  country  Afghanistan  AFG  region   
        1     175121.0  country  Afghanistan  AFG  region   
        2     175122.0  country  Afghanistan  AFG  region   
        3     175123.0  country  Afghanistan  AFG  region   
        4     175124.0  country  Afghanistan  AFG  region   
PM25    0     175089.0  country  Afghanistan  AFG  region   
        1     175104.0  country  Afghanistan  AFG  region   
        2     175105.0  country  Afghanistan  AFG  region   
        3     175106.0  country  Afghanistan  AFG  region   
        4     175107.0  country  Afghanistan  AFG  region   

                                   Name Exposure Lower Exposure Mean  \
Dataset                                                                
HAP     0  North Africa and Middle East           0.80          0.86   
        1  North Africa and Middle East           0.80          0.86   
        2  North Africa and Middle East           0.80          0.86   
        3  North Africa and Middle East           0.80          0.86   
        4  North Africa and Middle East           0.80          0.86   
NO2     0  North Africa and Middle East           0.95          12.3   
        1  North Africa and Middle East          0.806          12.1   
        2  North Africa and Middle East          0.636          11.7   
        3  North Africa and Middle East          0.443          11.4   
        4  North Africa and Middle East           0.23            11   
OZONE   0  North Africa and Middle East           55.8          56.6   
        1  North Africa and Middle East           54.6          55.4   
        2  North Africa and Middle East           53.2          54.1   
        3  North Africa and Middle East           52.7          53.6   
        4  North Africa and Middle East           53.1            54   
PM25    0  North Africa and Middle East           32.1          64.2   
        1  North Africa and Middle East           36.9          64.2   
        2  North Africa and Middle East           39.5          64.2   
        3  North Africa and Middle East           40.5          64.2   
        4  North Africa and Middle East           39.8          64.3   

          Exposure Upper  Year Pollutant    Pollutant Name  Region Name  \
Dataset                                                                   
HAP     0           0.90  1990       hap               HAP  GBD Regions   
        1           0.90  1991       hap               HAP  GBD Regions   
        2           0.90  1992       hap               HAP  GBD Regions   
        3           0.90  1993       hap               HAP  GBD Regions   
        4           0.90  1994       hap               HAP  GBD Regions   
NO2     0           12.3  1990       no2    NO<sub>2</sub>  GBD Regions   
        1           12.2  1991       no2    NO<sub>2</sub>  GBD Regions   
        2             12  1992       no2    NO<sub>2</sub>  GBD Regions   
        3           11.8  1993       no2    NO<sub>2</sub>  GBD Regions   
        4           11.6  1994       no2    NO<sub>2</sub>  GBD Regions   
OZONE   0           57.5  1990     ozone             Ozone  GBD Regions   
        1           56.4  1991     ozone             Ozone  GBD Regions   
        2             55  1992   

### 3.2 Checking Columns Similiarity
We can see in the below table that all 4 datasets have the very same columns.

In [6]:
# Collect columns for each dataframe
columns_dict = {
    "HAP": list(pollutant_data_hap.columns),
    "NO2": list(pollutant_data_no2.columns),
    "OZONE": list(pollutant_data_ozone.columns),
    "PM25": list(pollutant_data_pm25.columns)
}

# Create a DataFrame to display as a table
columns_df = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in columns_dict.items()]))
columns_df

,HAP,NO2,OZONE,PM25
0,Exposure Id,Exposure Id,Exposure Id,Exposure Id
1,Type,Type,Type,Type
2,Country,Country,Country,Country
3,ISO3,ISO3,ISO3,ISO3
4,Region,Region,Region,Region
5,Name,Name,Name,Name
6,Exposure Lower,Exposure Lower,Exposure Lower,Exposure Lower
7,Exposure Mean,Exposure Mean,Exposure Mean,Exposure Mean
8,Exposure Upper,Exposure Upper,Exposure Upper,Exposure Upper
9,Year,Year,Year,Year


In [7]:
# Display unique values for columns to justify removal
columns_to_check = [
    "Exposure Id", "Type", "ISO3", "Region", "Pollutant",
    "Pollutant Name", "Region Name", "Units"
]

datasets = {
    "HAP": pollutant_data_hap,
    "NO2": pollutant_data_no2,
    "OZONE": pollutant_data_ozone,
    "PM25": pollutant_data_pm25
}

for col in columns_to_check:
    print(f"Column: {col}")
    for name, df in datasets.items():
        unique_vals = df[col].unique()
        print(f"  {name}: {unique_vals[:5]}{' ...' if len(unique_vals) > 5 else ''} (n_unique={len(unique_vals)})")
    print()

# Show length of Year and Country columns to prove they are needed
for col in ["Year", "Country"]:
    print(f"Column: {col}")
    for name, df in datasets.items():
        print(f"  {name}: non-null count = {df[col].notnull().sum()}, unique count = {df[col].nunique()}")
    print()

Column: Exposure Id
  HAP: [175151. 175152. 175153. 175154. 175155.] ... (n_unique=6356)
  NO2: [175182. 175190. 175191. 175192. 175193.] ... (n_unique=6356)
  OZONE: [175120. 175121. 175122. 175123. 175124.] ... (n_unique=6356)
  PM25: [175089. 175104. 175105. 175106. 175107.] ... (n_unique=6356)

Column: Type
  HAP: ['country' '' 'citation'] (n_unique=3)
  NO2: ['country' '' 'citation'] (n_unique=3)
  OZONE: ['country' '' 'citation'] (n_unique=3)
  PM25: ['country' '' 'citation'] (n_unique=3)

Column: ISO3
  HAP: ['AFG' 'ALB' 'DZA' 'ASM' 'AND'] ... (n_unique=205)
  NO2: ['AFG' 'ALB' 'DZA' 'ASM' 'AND'] ... (n_unique=205)
  OZONE: ['AFG' 'ALB' 'DZA' 'ASM' 'AND'] ... (n_unique=205)
  PM25: ['AFG' 'ALB' 'DZA' 'ASM' 'AND'] ... (n_unique=205)

Column: Region
  HAP: ['region' 'country' nan] (n_unique=3)
  NO2: ['region' 'country' nan] (n_unique=3)
  OZONE: ['region' 'country' nan] (n_unique=3)
  PM25: ['region' 'country' nan] (n_unique=3)

Column: Pollutant
  HAP: ['hap' nan] (n_unique=2)
 

## 4. Dataset Analysis & Preparation Strategy of Pollutants Data

All available datasets exhibit a consistent structural format, which simplifies the process of merging them into a unified Pollutants Dataset. Consolidating these datasets will enhance the efficiency and coherence of our analytical workflow.

### Column Removal Strategy

Based on our analysis, we identify and remove several columns that do not contribute meaningfully to our analysis:

- **Exposure Id**: Serves as a unique identifier; does not offer analytical value.
- **Type**: Contains a single constant value, "Country", indicating the level of aggregation—provides no variability or insight.
- **ISO3**: Merely reiterates the country designation in ISO3 format; redundant alongside the full country name.
- **Region**: Contains only two values, distinguishing whether a record pertains to a region or country—information already implied or unneeded in our use case.
- **Pollutant**: Although it specifies the pollutant type, we will disaggregate exposure metrics by pollutant, making this column unnecessary.
- **Pollutant Name**: A duplicate of the Pollutant column, thus redundant.
- **Region Name**: Uniformly states "GDB Region" for relevant rows; offers no analytical distinction.
- **Units**: Indicates the measurement units for pollutants. While relevant from a reporting standpoint, the unit is consistent across datasets and will be addressed in the methodological literature. Hence, its repetition in every row is unnecessary.

### Column Refactoring

The following columns will be renamed to enhance semantic clarity and analytical alignment:
- **Name** --> **Region Name**: More accurately reflects the contents of the column, which specifies regional labels.
- **Exposure Lower** --> **Exposure Lower [Pollutant]**: Specifies that the value represents the lower bound of average exposure for a given pollutant.
- **Exposure Mean** --> **Exposure Mean [Pollutant]**: Clarifies that the value denotes the average (mean) exposure rate.
- **Exposure Upper** --> **Exposure Upper [Pollutant]**: Indicates the upper bound of average exposure for the pollutant in question.

### Columns Retained Without Modification

The following fields will be preserved in their original form due to their essential role in analysis:
- **Country**: Contains country names; critical for spatial analysis.
- **Year**: Records the year of data collection; necessary for temporal trend analysis.

In [8]:
# Define a function to clean and prepare each dataset
def prepare_dataset(df, pollutant_name):
    """
    Process a pollutant dataset by:
    1. Removing unnecessary columns
    2. Renaming columns to include pollutant type
    """
    # Drop unnecessary columns
    columns_to_drop = [
        "Exposure Id", "Type", "ISO3", "Region", "Pollutant",
        "Pollutant Name", "Region Name", "Units"
    ]
    df = df.drop(columns=columns_to_drop, errors='ignore')
    
    # Rename columns
    df = df.rename(columns={
        "Name": "Region Name",
        "Exposure Lower": f"Exposure Lower {pollutant_name}",
        "Exposure Mean": f"Exposure Mean {pollutant_name}",
        "Exposure Upper": f"Exposure Upper {pollutant_name}"
    })
    
    return df

# Prepare each dataset
hap_cleaned = prepare_dataset(pollutant_data_hap, "HAP")
no2_cleaned = prepare_dataset(pollutant_data_no2, "NO2")
ozone_cleaned = prepare_dataset(pollutant_data_ozone, "OZONE")
pm25_cleaned = prepare_dataset(pollutant_data_pm25, "PM25")

## 5. Data Merging
Now that we've cleaned and verified each dataset, we'll concatenate all four datasets into a single comprehensive DataFrame. This creates a unified dataset for subsequent analysis.

In [9]:
# Merge datasets on common columns (Country, Year, Region Name)
# Start with HAP dataset
merged_data = hap_cleaned

# Merge with NO2 dataset
merged_data = pd.merge(merged_data, no2_cleaned, on=["Country", "Year", "Region Name"], how="inner")
print(f"After merging NO2 - shape: {merged_data.shape}")

# Merge with OZONE dataset
merged_data = pd.merge(merged_data, ozone_cleaned, on=["Country", "Year", "Region Name"], how="inner")
print(f"After merging OZONE - shape: {merged_data.shape}")

# Merge with PM25 dataset
merged_data = pd.merge(merged_data, pm25_cleaned, on=["Country", "Year", "Region Name"], how="inner")
print(f"Final merged shape: {merged_data.shape}")

# Preview the merged dataset
merged_data.head()

After merging NO2 - shape: (6357, 9)
After merging OZONE - shape: (6357, 12)
Final merged shape: (6357, 15)


,Country,Region Name,Exposure Lower HAP,Exposure Mean HAP,Exposure Upper HAP,Year,Exposure Lower NO2,Exposure Mean NO2,Exposure Upper NO2,Exposure Lower OZONE,Exposure Mean OZONE,Exposure Upper OZONE,Exposure Lower PM25,Exposure Mean PM25,Exposure Upper PM25
0,Afghanistan,North Africa and Middle East,0.80,0.86,0.90,1990,0.95,12.3,12.3,55.8,56.6,57.5,32.1,64.2,115
1,Afghanistan,North Africa and Middle East,0.80,0.86,0.90,1991,0.806,12.1,12.2,54.6,55.4,56.4,36.9,64.2,104
2,Afghanistan,North Africa and Middle East,0.80,0.86,0.90,1992,0.636,11.7,12,53.2,54.1,55,39.5,64.2,95.7
3,Afghanistan,North Africa and Middle East,0.80,0.86,0.90,1993,0.443,11.4,11.8,52.7,53.6,54.4,40.5,64.2,94.2
4,Afghanistan,North Africa and Middle East,0.80,0.86,0.90,1994,0.23,11,11.6,53.1,54,54.9,39.8,64.3,99.9


## 6. Missing Value Detection and Handling:
During the merging process, we identified some rows with NaN values. These appear to be citation rows from the original CSV files that were loaded during import. Since these rows contain no meaningful data for our analysis, we remove them to ensure data integrity in the merged dataset.

In [10]:
# Show count of null/NA values for each column in merged_data
null_counts = merged_data.isnull().sum()
print("Null/NA values per column:")
print(null_counts)

Null/NA values per column:
Country                 0
Region Name             2
Exposure Lower HAP      2
Exposure Mean HAP       2
Exposure Upper HAP      2
Year                    2
Exposure Lower NO2      2
Exposure Mean NO2       2
Exposure Upper NO2      2
Exposure Lower OZONE    2
Exposure Mean OZONE     2
Exposure Upper OZONE    2
Exposure Lower PM25     2
Exposure Mean PM25      2
Exposure Upper PM25     2
dtype: int64


In [11]:
nan_rows_count = merged_data.isnull().any(axis=1).sum()
print(f"Number of rows with any NaN: {nan_rows_count}")

Number of rows with any NaN: 2


In [12]:
# Drop rows with any NaN values (the citation rows) from merged_data
merged_data = merged_data.dropna(how='any')
print(f"Shape after dropping NaN rows: {merged_data.shape}")

Shape after dropping NaN rows: (6355, 15)


## 7. Data Type Validation and Conversion

In [13]:
# Display a table of column names and their data types for merged_data
col_dtype_df = pd.DataFrame({
    'Column Name': merged_data.columns,
    'Data Type': merged_data.dtypes.astype(str).values
})
col_dtype_df

,Column Name,Data Type
0,Country,object
1,Region Name,object
2,Exposure Lower HAP,object
3,Exposure Mean HAP,object
4,Exposure Upper HAP,object
5,Year,object
6,Exposure Lower NO2,object
7,Exposure Mean NO2,object
8,Exposure Upper NO2,object
9,Exposure Lower OZONE,object


In [14]:
# Convert categorical variables to 'category' dtype
categorical_columns = ['Country', 'Region Name', 'Year']

for col in categorical_columns:
    if col in merged_data.columns:
        merged_data[col] = merged_data[col].astype('category')

# Convert numerical exposure variables to float dtype
numerical_columns = [col for col in merged_data.columns if any(x in col for x in ['Exposure Lower', 'Exposure Mean', 'Exposure Upper'])]

for col in numerical_columns:
    if col in merged_data.columns:
        merged_data[col] = pd.to_numeric(merged_data[col], errors='coerce')

# Display updated data types
col_dtype_df = pd.DataFrame({
    'Column Name': merged_data.columns,
    'Data Type': merged_data.dtypes.astype(str).values
})
col_dtype_df

,Column Name,Data Type
0,Country,category
1,Region Name,category
2,Exposure Lower HAP,float64
3,Exposure Mean HAP,float64
4,Exposure Upper HAP,float64
5,Year,category
6,Exposure Lower NO2,float64
7,Exposure Mean NO2,float64
8,Exposure Upper NO2,float64
9,Exposure Lower OZONE,float64


## 8. Outlier Detection

Outlier detection for environmental time series data requires special consideration since pollution levels naturally change over time due to industrialization, policy changes, and other factors. Using traditional methods like IQR across the entire many valid data points a dataset can incorrectly flags outliers simply because they represent a different time period.

### 8.1 Time-Aware Outlier Detection Approaches

Given that our dataset has yearly granularity with only one observation per country per year, we need detection approaches that account for this structure. We'll implement and compare two more appropriate methods:

1. **Year-Specific Comparison**: For each year, compares pollution levels across all countries, identifying countries with unusually high or low values compared to global patterns for that specific year.

2. **Country-Specific Temporal Deviation**: Analyzes each country's historical pattern and identifies years that deviate significantly from that country's expected temporal trend.

These approaches recognize both the yearly granularity of our data and the importance of considering temporal trends when identifying genuine outliers in environmental data.

In [15]:
# Year-Grouped Method: Detect outliers within each year separately compared to global levels at that time.
def detect_year_grouped_outliers(df, columns, z_threshold=3.0):
    outlier_indices = {}
    total_outliers = {}
    
    # Initialize total outlier count dictionary
    for col in columns:
        total_outliers[col] = 0
        outlier_indices[col] = []
    
    # Group by Year
    for year, year_df in df.groupby('Year', observed=False ):
        for col in columns:
            # Use z-score method for each year
            z_scores = np.abs(stats.zscore(year_df[col], nan_policy='omit'))
            outliers = year_df[z_scores > z_threshold]
            
            # Store outlier indices
            if len(outliers) > 0:
                outlier_indices[col].extend(outliers.index.tolist())
                total_outliers[col] += len(outliers)
    
    # Print summary of outliers found
    for col in columns:
        print(f"{col}: {total_outliers[col]} outliers detected across all years")
        
    return outlier_indices

# Rolling Window Z-score Method
def detect_rolling_window_outliers(df, columns, window_size=5, z_threshold=3.0):
    """
    Detect outliers using a rolling window approach.
    For each data point, computes z-score relative to surrounding points.
    """
    outlier_indices = {}
    total_outliers = {}
    
    # Initialize total outlier count dictionary
    for col in columns:
        total_outliers[col] = 0
        outlier_indices[col] = []
    
    # Group by Country 
    for country, country_df in df.groupby('Country', observed=False ):
        # Sort by year within each country
        country_df = country_df.sort_values('Year')
        
        for col in columns:
            # Skip if too few data points
            if len(country_df) < window_size:
                continue
                
            # Calculate rolling mean and std
            rolling_mean = country_df[col].rolling(window=window_size, center=True, min_periods=3).mean()
            rolling_std = country_df[col].rolling(window=window_size, center=True, min_periods=3).std()
            
            # Calculate z-scores
            z_scores = np.abs((country_df[col] - rolling_mean) / rolling_std)
            
            # Identify outliers
            outliers = country_df[z_scores > z_threshold]
            
            # Store outlier indices
            if len(outliers) > 0:
                outlier_indices[col].extend(outliers.index.tolist())
                total_outliers[col] += len(outliers)
    
    # Print summary of outliers found
    for col in columns:
        print(f"{col}: {total_outliers[col]} outliers detected using rolling window method")
        
    return outlier_indices

# Apply all three methods and compare results
print("1. Year-Grouped Outlier Detection:")
year_grouped_outlier_indices = detect_year_grouped_outliers(merged_data, numerical_columns[:3])  # Using first 3 columns for clarity

print("\n2. Rolling Window Z-score Outlier Detection:")
rolling_window_outlier_indices = detect_rolling_window_outliers(merged_data, numerical_columns[:3])


1. Year-Grouped Outlier Detection:
Exposure Lower HAP: 0 outliers detected across all years
Exposure Mean HAP: 0 outliers detected across all years
Exposure Upper HAP: 0 outliers detected across all years

2. Rolling Window Z-score Outlier Detection:
Exposure Lower HAP: 0 outliers detected using rolling window method
Exposure Mean HAP: 0 outliers detected using rolling window method
Exposure Upper HAP: 0 outliers detected using rolling window method


Considering the dataset was gathered from reputable source, it was very well expected that there would be 0 outliers in the data. 

## 9. Duplicate Detection


In [16]:
# Check for duplicates based on Country, Year, and Region Name
duplicates_mask = merged_data.duplicated(subset=['Country', 'Year'], keep=False)

# Extract all duplicate entries as a dataframe
duplicates_df = merged_data[duplicates_mask]

# Print the duplicate entries dataframe
print(f"Number of duplicate rows: {len(duplicates_df)}")
duplicates_df

Number of duplicate rows: 62


,Country,Region Name,Exposure Lower HAP,Exposure Mean HAP,Exposure Upper HAP,Year,Exposure Lower NO2,Exposure Mean NO2,Exposure Upper NO2,Exposure Lower OZONE,Exposure Mean OZONE,Exposure Upper OZONE,Exposure Lower PM25,Exposure Mean PM25,Exposure Upper PM25
5952,United Kingdom,United Kingdom,0.0,0.0,0.0,1990,6.600,24.2,19.3,33.9,34.2,34.6,10.50,18.70,31.3
5953,United Kingdom,Western Europe,0.0,0.0,0.0,1990,6.600,24.2,19.3,33.9,34.2,34.6,10.50,18.70,31.3
5954,United Kingdom,Western Europe,0.0,0.0,0.0,1991,6.480,24.0,19.2,33.6,33.9,34.3,12.20,18.50,28.0
5955,United Kingdom,United Kingdom,0.0,0.0,0.0,1991,6.480,24.0,19.2,33.6,33.9,34.3,12.20,18.50,28.0
5956,United Kingdom,Western Europe,0.0,0.0,0.0,1992,6.360,23.8,19.1,32.3,32.6,32.9,12.80,18.40,25.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6009,United Kingdom,United Kingdom,0.0,0.0,0.0,2018,2.180,15.9,14.9,31.8,32.1,32.5,9.91,10.20,10.6
6010,United Kingdom,Western Europe,0.0,0.0,0.0,2019,1.600,14.8,14.3,31.6,32.0,32.3,9.61,9.95,10.3
6011,United Kingdom,United Kingdom,0.0,0.0,0.0,2019,1.600,14.8,14.3,31.6,32.0,32.3,9.61,9.95,10.3
6012,United Kingdom,Western Europe,0.0,0.0,0.0,2020,-0.332,11.2,12.4,31.5,31.8,32.1,9.20,9.91,10.6


It appears that the United Kingdom is listed twice in the dataset. Once under the Western Europe region and once as a standalone United Kingdom region. This duplication likely comes from the data collection method: Since the filters applied included both “All Other Countries” and “Current Country” set to United Kingdom, the country may have been captured in both contexts. To address this inconsistency, the most reliable solution is to remove the rows where both the Region and Country fields are set to United Kingdom.

In [17]:
# Check number of rows where "Region Name" == "United Kingdom"
uk_region_rows = merged_data[merged_data['Region Name'] == "United Kingdom"]
print(f"Number of rows with Region Name = 'United Kingdom': {len(uk_region_rows)}")

Number of rows with Region Name = 'United Kingdom': 31


In [18]:
# Delete those rows
merged_data = merged_data[merged_data['Region Name'] != "United Kingdom"]
print(f"Shape after deleting 'United Kingdom' region rows: {merged_data.shape}")

Shape after deleting 'United Kingdom' region rows: (6324, 15)


## 10. Export Processed Dataset

Saving the final consolidated and processed dataset to a CSV file for future analysis. This file will be the input for subsequent analytical work.

In [19]:
# Save both original and cleaned datasets
merged_data.to_csv('merged_pollutants_data.csv', index=False)
print(f"Saved original merged data to 'merged_pollutants_data.csv'")

Saved original merged data to 'merged_pollutants_data.csv'


## Conclusion

This notebook has documented a comprehensive pre-processing workflow for multiple global air pollutant datasets, transforming raw data into a cohesive, analysis-ready format. The process began with the acquisition of four distinct pollutant datasets containing Household Air Pollution (HAP), Nitrogen Dioxide (NO2), Ozone, and Particulate Matter 2.5 (PM2.5) each having valuable environmental exposure metrics across different countries and years.

Upon initial examination, we identified numerous data quality issues that required systematic attention. The datasets contained Excel-style formatting artifacts, with values enclosed in equals signs and quotes (`="..."`), which were systematically removed through a custom cleaning function. This standardization was essential for ensuring consistent interpretation of the data values during subsequent analysis.

Our exploratory data analysis revealed a uniform structure across all four datasets, with identical column configurations but varying exposure values specific to each pollutant type. This consistency facilitated the development of a streamlined data preparation strategy. Through careful column analysis, we identified and eliminated eight redundant or uninformative fields that added no analytical value, including exposure IDs, redundant country codes, and static descriptors. Column names were then semantically enhanced to clearly indicate the pollutant type for each exposure measurement, improving interpretability while maintaining the essential spatial-temporal references (country, region, year).

The merging process combined all four pollutant datasets along their common dimensions (country, year, and region), creating a unified dataset that enables integrated analysis of multiple pollutant exposures. During this consolidation, we identified and addressed several data integrity issues. Citation rows containing missing values were systematically removed, and data types were appropriately converted to optimize storage and processing efficiency—categorical variables were converted to category types, while numerical exposure values were ensured to be in proper floating-point format.

To maintain data quality, we implemented sophisticated time-aware outlier detection methods tailored to the yearly granularity of our environmental data. Rather than applying IQR thresholds, we employed year-grouped and rolling window approaches that respect the temporal nature of pollution trends. Interestingly, no significant outliers were detected, confirming the high quality of the source data.

During duplicate detection, we discovered and resolved an inconsistency where the United Kingdom appeared twice in the dataset—once as part of Western Europe and once as a standalone region. This duplication was systematically addressed by removing the rows where both region and country were set to "United Kingdom," maintaining data integrity without information loss.

The result of this pre-processing workflow is a single, consolidated CSV file (`merged_pollutants_data.csv`) that combines exposure metrics for all four pollutants across multiple countries and years. This dataset is now optimized for subsequent analytical work, featuring standardized formatting, appropriate data types, resolved duplications, and comprehensive pollutant exposure metrics. The pre-processed dataset forms a robust foundation for exploring relationships between air pollution exposures and health outcomes, supporting evidence-based environmental health research and policy development.